# parsing.ms_office.markitdown.win

> Microsoft Office document parsing with markitdown on Windows Platform

In [ ]:
# |default_exp parsing.ms_office.markitdown.win

In [ ]:
# | hide
from nbdev.showdoc import *
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

## Setup

Install the notebook dependencies in the active Windows Jupyter kernel if needed:

```powershell
python -m pip install "markitdown[all]" python-dotenv
```

Set `OFFICE_FILES_ROOT` in `PROJ_ROOT/.env`. Prefer a forward-slash Windows path, for example `OFFICE_FILES_ROOT=D:/documents/office`.

In [ ]:
# | export
import base64
import binascii
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path, PurePosixPath

from dotenv import load_dotenv

In [ ]:
#| export
def _find_project_root(start: Path | str | None = None) -> Path:
    """Find PROJ_ROOT from the environment or a parent pyproject.toml."""
    configured_root = os.getenv("PROJ_ROOT")
    if configured_root:
        project_root = Path(configured_root).expanduser().resolve()
        if not project_root.is_dir():
            raise FileNotFoundError(f"PROJ_ROOT is not a directory: {project_root}")
        return project_root

    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"Could not find pyproject.toml above {current}")


PROJ_ROOT = _find_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Project environment file not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)


def get_office_files_root() -> Path:
    """Read and validate OFFICE_FILES_ROOT from PROJ_ROOT/.env."""
    configured_root = os.getenv("OFFICE_FILES_ROOT")
    if not configured_root:
        raise RuntimeError(f"OFFICE_FILES_ROOT is not configured in {ENV_FILE}")

    root = Path(configured_root).expanduser()
    if not root.is_absolute():
        root = PROJ_ROOT / root
    root = root.resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"OFFICE_FILES_ROOT is not a directory: {root}")
    return root

In [ ]:
#| export
OFFICE_EXTENSIONS = frozenset({
    ".doc", ".docx", ".odt",
    ".ppt", ".pptx", ".odp",
    ".xls", ".xlsx", ".xlsm", ".xlsb", ".ods",
    ".csv", ".tsv",
})

IMAGE_EXTENSION_BY_MIME = {
    "jpeg": ".jpg",
    "jpg": ".jpg",
    "png": ".png",
    "gif": ".gif",
    "webp": ".webp",
    "bmp": ".bmp",
    "tiff": ".tiff",
    "svg+xml": ".svg",
    "wmf": ".wmf",
    "x-wmf": ".wmf",
    "emf": ".emf",
    "x-emf": ".emf",
}

MARKDOWN_DATA_IMAGE_RE = re.compile(
    r"(?P<prefix>!\[[^\]]*\]\(\s*)data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)"
    r"(?P<suffix>\s*\))",
    flags=re.IGNORECASE,
)
HTML_DATA_IMAGE_RE = re.compile(
    r"(?P<prefix><img\b[^>]*?\bsrc\s*=\s*(?P<quote>[\"']))data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)"
    r"(?P<suffix>(?P=quote)[^>]*>)",
    flags=re.IGNORECASE,
)

In [ ]:
#| export
def convert_office_to_md(
    root_folder: Path | str,
    output_root: Path | str | None = None,
    *,
    overwrite: bool = False,
) -> dict[str, object]:
    """
    Recursively convert supported Office files to Markdown with MarkItDown.

    Output mirrors the input tree under ``output_root``. Each source document
    gets a folder containing ``<stem>.md`` and, after extraction, ``img/``.
    """
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Office root is not a directory: {root}")

    md_root = Path(output_root).expanduser() if output_root else root / ".md"
    if not md_root.is_absolute():
        md_root = root / md_root
    md_root = md_root.resolve()
    md_root.mkdir(parents=True, exist_ok=True)

    source_files = sorted(
        path for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in OFFICE_EXTENSIONS
        and not path.name.startswith("~$")
    )
    report = {
        "root": root,
        "output_root": md_root,
        "discovered": source_files,
        "converted": [],
        "skipped": [],
        "failed": [],
    }

    for source in source_files:
        relative_source = source.relative_to(root)
        markdown_file = (
            md_root / relative_source.parent / source.stem / f"{source.stem}.md"
        )
        if markdown_file.exists() and not overwrite:
            report["skipped"].append(markdown_file)
            print(f"Skipped existing: {markdown_file}")
            continue

        markdown_file.parent.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable,
            "-m",
            "markitdown",
            str(source),
            "-o",
            str(markdown_file),
            "--keep-data-uris",
        ]
        try:
            subprocess.run(command, check=True)
        except (OSError, subprocess.CalledProcessError) as error:
            report["failed"].append((source, error))
            print(f"Failed: {source}: {error}")
            continue

        report["converted"].append(markdown_file)
        print(f"Converted: {source} -> {markdown_file}")

    return report

In [ ]:
#| hide
# MarkItDown runs through the active notebook interpreter, which avoids PATH
# ambiguity when several Python environments are installed on Windows.

In [ ]:
#| hide
def _extract_md_base64_images_win_legacy(markdown_file_path, image_output_folder=".") -> int:
    """
    Extracts base64 embedded images from a Markdown file, saves them to a folder,
    and replaces the base64 strings with relative paths to the new image files.
    This version is tailored for Windows environments, ensuring compatibility
    with Windows file paths and handling of WMF/EMF images using ImageMagick 'magick' command.

    Args:
        markdown_file_path (str): Path to the input Markdown file.
        image_output_folder (str): Name of the folder to save extracted images.
                                   This folder will be created relative to the
                                   Markdown file's directory if it doesn't exist.
    """
    if not os.path.exists(markdown_file_path):
        print(f"Error: Markdown file not found at {markdown_file_path}")
        return -1
    # markdown_file_stem = markdown_file_path.stem
    markdown_dir = os.path.dirname(os.path.abspath(markdown_file_path))
    full_image_output_path = os.path.join(markdown_dir, image_output_folder)

    if not os.path.exists(full_image_output_path):
        os.makedirs(full_image_output_path)
        # print(f"Created image output folder: {full_image_output_path}")

    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Regex to find base64 encoded images in Markdown
    # Pattern: ![alt text](data:image/png;base64,BASE64_STRING)
    # Groups:
    # 1: Alt text
    # 2: Image format (e.g., png, jpeg)
    # 3: Base64 data string
    # We also capture the full match (group 0) to replace it
    regex_md_img_quote = r"!\[(.*?)\]\(data:image/(.+?);base64,([A-Za-z0-9+/=\s]+)\)"
    regex_illegal_file_name = r'[^a-zA-Z0-9_\-\.]+'  # Legal characters for filenames

    first_match = re.search(regex_md_img_quote, content)
    if not first_match:
        return -1
    new_content = content
    images_extracted_count = 0

    # We need to iterate carefully as string replacements change string length
    # Finding all matches first and then replacing is safer, but can be tricky
    # if matches overlap (not typical for this pattern).
    # A simpler approach for non-overlapping, distinct matches is to iterate
    # and replace. For more complex scenarios, one might work on a list of lines
    # or use re.sub with a function.

    # Using re.finditer to get match objects for more control
    for i, match in enumerate(re.finditer(regex_md_img_quote, content)):
        full_match_str = match.group(0)
        alt_text = match.group(1)
        # Normalize alt text to a legal filename
        alt_text = re.sub(regex_illegal_file_name, '_', alt_text)  # Replace illegal characters with '_'
        alt_text = alt_text.strip()  # Remove leading/trailing whitespace
        alt_text = alt_text[:50] if len(alt_text) > 50 else alt_text  # Limit length to 50 characters
        alt_text = 'img' if not alt_text else alt_text # If alt text is empty, use a default name

        image_format = match.group(2).lower() # e.g., png, jpeg
        image_format = re.sub(r'x-([a-zA-Z])', r'\1', image_format) # Normalize format (e.g., x-wmf/x-emf to wmf/emf)
        base64_data = match.group(3)

        # Clean up base64 data (remove potential whitespace)
        base64_data_cleaned = "".join(base64_data.split())
        # Fix missing padding
        missing_padding = len(base64_data_cleaned) % 4
        if missing_padding != 0:
            base64_data_cleaned += '=' * (4 - missing_padding)
        try:
            image_data = base64.b64decode(base64_data_cleaned)
        except base64.binascii.Error as e:
            print(f"Warning: Could not decode base64 string for an image (alt: {alt_text}). Error: {e}")
            continue # Skip this image

        # Generate a unique filename
        # Using a counter is simple, could use uuid for more robustness
        image_filename = f"{alt_text}_{images_extracted_count}.{image_format}"
        image_filepath = os.path.join(full_image_output_path, image_filename)

        # Save the image
        with open(image_filepath, 'wb') as img_file:
            img_file.write(image_data)
        # print(f"Extracted and saved: {image_filepath}")
        if image_format == 'wmf': # in case of wmf, we need to convert it to svg with soffice
            svg_file = Path(image_filepath).with_suffix('.svg') 
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(image_filepath),
                '-density', '300',
                '-trim', '-border', '5',
                str(svg_file),
            ], check=True)
            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(image_filepath),
                '-density', '300',
                '-trim', '-border', '5',
                str(png_file),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        if image_format == 'emf': # in case of wmf, we need to convert it to svg with soffice
            svg_file = Path(image_filepath).with_suffix('.svg') 
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(image_filepath),
                '-density', '300',
                '-trim', '-border', '5',
                str(svg_file),
            ], check=True)
            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(image_filepath),
                '-density', '300',
                '-trim', '-border', '5',
                str(png_file),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        if image_format == 'gif': # in case of gif, we need to convert it to png with imagemagick convert
            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(image_filepath),
                '-density', '300',
                '-trim', '-border', '5',
                str(png_file),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        
        # Create the new Markdown image link (relative path)
        # The path in Markdown should be relative to the Markdown file itself
        relative_image_path = os.path.join(image_output_folder, image_filename)
        # Ensure forward slashes for Markdown paths, even on Windows
        relative_image_path_markdown = relative_image_path.replace(os.sep, '/')
        new_image_md_link = f"![{alt_text}]({relative_image_path_markdown})"

        # Replace the original base64 string with the new link in the `new_content`
        # Only replace the first occurrence of this specific full_match_str in case of duplicates
        # (though each match from finditer is unique in its position)
        new_content = new_content.replace(full_match_str, new_image_md_link, 1)
        images_extracted_count += 1

    if images_extracted_count > 0:
        # Save the modified Markdown content
        # You might want to save to a new file, e.g., original_filename_modified.md
        # For this example, I'll overwrite the original. Be careful!
        # Consider backing up your original file first.
        output_markdown_file_path = markdown_file_path # Overwrite
        # output_markdown_file_path = os.path.splitext(markdown_file_path)[0] + "_modified.md" # New file

        with open(output_markdown_file_path, 'w', encoding='utf-8') as f:
            f.write(new_content)

    return images_extracted_count


In [ ]:
#| export
def convert_md_gif2png_win(markdown_file_path, image_output_folder=".") -> int:
    """
    Convert gif image to png files and replace the image link with png for llm processing 
    This version is tailored for Windows environments, ensuring compatibility
    with Windows file paths and handling of WMF/EMF images using ImageMagick 'magick' command.

    Args:
        markdown_file_path (str): Path to the input Markdown file.
        image_output_folder (str): Name of the folder to save extracted images.
                                This folder will be created relative to the
                                Markdown file's directory if it doesn't exist.
    """
    if not os.path.exists(markdown_file_path):
        print(f"Error: Markdown file not found at {markdown_file_path}")
        return -1
    # markdown_file_stem = markdown_file_path.stem
    markdown_dir = os.path.dirname(os.path.abspath(markdown_file_path))
    full_image_output_path = os.path.join(markdown_dir, image_output_folder)

    if not os.path.exists(full_image_output_path):
        os.makedirs(full_image_output_path)
        # print(f"Created image output folder: {full_image_output_path}")

    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Regex to find base64 encoded images in Markdown
    # Pattern: ![alt text](BASE64_STRING.gif)
    ## Pattern: ![alt text](data:image/png;base64,BASE64_STRING)
    # Groups:
    # 1: Alt text
    # 2: Image Path and name (with '/')
    # 3: the last (.*?) as Image format (gif)
    # We also capture the full match (group 0) to replace it
    # regex_md_img_quote = r"!\[(.*?)\]\(data:image/(.+?);base64,([A-Za-z0-9+/=\s]+)\)"
    # regex_md_img_quote = r"!\[(.*?)\]\(([A-Za-z0-9+/=\s]+)\.(.+?)\)"
    # regex_md_img_quote = r"!\[(.*?)\]\(([A-Za-z0-9+/=\s]+)\.(.{3,4})\)"
    regex_md_img_quote = r"!\[(.*?)\]\((.*?)\.(gif)\)"
    regex_legal_file_name = r'[^a-zA-Z0-9_\-\.]+'  # Legal characters for filenames

    first_match = re.search(regex_md_img_quote, content)
    if not first_match:
        return -1
    new_content = content
    images_extracted_count = 0

    # We need to iterate carefully as string replacements change string length
    # Finding all matches first and then replacing is safer, but can be tricky
    # if matches overlap (not typical for this pattern).
    # A simpler approach for non-overlapping, distinct matches is to iterate
    # and replace. For more complex scenarios, one might work on a list of lines
    # or use re.sub with a function.

    # Using re.finditer to get match objects for more control
    for i, match in enumerate(re.finditer(regex_md_img_quote, content)):
        full_match_str = match.group(0)
        alt_text = match.group(1)
        # Normalize alt text to a legal filename
        alt_text = re.sub(regex_legal_file_name, '_', alt_text)  # Replace illegal characters with '_'
        alt_text = alt_text.strip()  # Remove leading/trailing whitespace
        alt_text = alt_text[:50] if len(alt_text) > 50 else alt_text  # Limit length to 50 characters
        alt_text = 'img' if not alt_text else alt_text # If alt text is empty, use a default name

        # image_format = match.group(2).lower() # e.g., png, jpeg
        # image_format = re.sub(r'x-([a-zA-Z])', r'\1', image_format) # Normalize format (e.g., x-wmf/x-emf to wmf/emf)
        # base64_data = match.group(3)
        image_data_path = match.group(2)
        image_format = match.group(3).lower() # e.g., gif

        # Generate a unique filename
        # Using a counter is simple, could use uuid for more robustness
        image_filename = f"{image_data_path}.{image_format}"
        image_filepath = os.path.join(full_image_output_path, image_filename)

        if image_format == 'gif': # in case of gif, we need to convert it to png with imagemagick convert
            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(image_filepath),
                '-density', '300',
                '-trim', '-border', '5',
                str(png_file),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        
        # Create the new Markdown image link (relative path)
        # The path in Markdown should be relative to the Markdown file itself
        relative_image_path = os.path.join(image_output_folder, image_filename)
        # Ensure forward slashes for Markdown paths, even on Windows
        relative_image_path_markdown = relative_image_path.replace(os.sep, '/')
        new_image_md_link = f"![{alt_text}]({relative_image_path_markdown})"

        # Replace the original base64 string with the new link in the `new_content`
        # Only replace the first occurrence of this specific full_match_str in case of duplicates
        # (though each match from finditer is unique in its position)
        new_content = new_content.replace(full_match_str, new_image_md_link, 1)
        images_extracted_count += 1

    if images_extracted_count > 0:
        # Save the modified Markdown content
        # You might want to save to a new file, e.g., original_filename_modified.md
        # For this example, I'll overwrite the original. Be careful!
        # Consider backing up your original file first.
        output_markdown_file_path = markdown_file_path # Overwrite
        # output_markdown_file_path = os.path.splitext(markdown_file_path)[0] + "_modified.md" # New file

        with open(output_markdown_file_path, 'w', encoding='utf-8') as f:
            f.write(new_content)

    return images_extracted_count


In [ ]:
#| export
def convert_gif2png_from_md(root_folder):
    """
    Recursively convert all .md files under
    root_folder (and subfolders) and extract all base64 images in them into
    a separate folder and replace the base64 image references in the markdown
    files with the path to the extracted image.
    """


    root = Path(root_folder)
    # tmp = Path(root_folder).parent / 'tmp'
    # shutil.move(root, tmp) # Copy the whole folder to res
    # os.makedirs(root, exist_ok=False)  # make sure the root folder exists and is empty
    image_folder = "img"
    for file in root.rglob('*'):
        if file.suffix.lower() == '.md':
            try:
                # Create a folder with the file name and move the md file into it
                md_folder = file.parent  # / (file.stem)
                # res_folder = root / md_folder.relative_to(tmp)
                # shutil.copy(md_folder, res_folder)
                # new_md_file = res_folder / file.name
                # Use replace original md file with base64 extracted in separate image folder
                res = convert_md_gif2png_win(file,image_folder)
                if res == -1:
                    print(f"Not Converted: {file}")
                else:
                    print(f'Converted {res} images in {file}')
                # file.unlink()  # Remove the original md file after extraction
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {file}: {e}")
    
    # shutil.rmtree(tmp)  # Remove the temporary folder after extraction
# Optional: convert_gif2png_from_md(PROCESS_REPORT["conversion"]["output_root"])

In [ ]:
#| export
def extract_md_html_images_win(markdown_file_path) -> int:
    """
    Extracts base64 embedded images from a Markdown file, saves them to a folder,
    and replaces the base64 strings with relative paths to the new image files.
    This version is tailored for Windows environments, ensuring compatibility
    with Windows file paths and handling of WMF/EMF images using ImageMagick 'magick' command.

    Args:
        markdown_file_path (str): Path to the input Markdown file.
        image_output_folder (str): Name of the folder to save extracted images.
                                This folder will be created relative to the
                                Markdown file's directory if it doesn't exist.
    """
    if not os.path.exists(markdown_file_path):
        # print(f"Error: Markdown file not found at {markdown_file_path}")
        return -1

    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Regex to find base64 encoded images in Markdown
    # Pattern: ![alt text](data:image/png;base64,BASE64_STRING)
    # Groups:
    # 1: Alt text
    # 2: Image format (e.g., png, jpeg)
    # 3: Base64 data string
    # We also capture the full match (group 0) to replace it
    # regex_md_img_quote = r"!\[(.*?)\]\(data:image/(.+?);base64,([A-Za-z0-9+/=\s]+)\)"
    # regex_html_img_quote = r'<img[^>]*src\s*=\s*["\']([^"\']+)["\'][^>]*>'  # Matches HTML img tags with src attributes
    regex_html_img_quote = r'<img[^>]*src\s*=\s*["\']([^"\']+/(media/.+\.([a-z]{3,4})))["\'][^>]*>'  # Matches HTML img tags with src attributes
    # regex_illegal_file_name = r'[^a-zA-Z0-9_\-\.]+'  # Legal characters for filenames

    first_match = re.search(regex_html_img_quote, content)
    if not first_match:
        return -1

    new_content = content
    images_extracted_count = 0

    # We need to iterate carefully as string replacements change string length
    # Finding all matches first and then replacing is safer, but can be tricky
    # if matches overlap (not typical for this pattern).
    # A simpler approach for non-overlapping, distinct matches is to iterate
    # and replace. For more complex scenarios, one might work on a list of lines
    # or use re.sub with a function.

    # Using re.finditer to get match objects for more control
    png_file_path = svg_file_path = None
    for i, match in enumerate(re.finditer(regex_html_img_quote, content)):
        # full_match_str = match.group(0)
        # Extract the file path and suffix from the match
        absolute_file_path = match.group(1)  # e.g., /absolute/path/to/media/image.png
        relative_file_path = Path('./' + match.group(2))  # e.g., ./media/image.png
        local_file_path = Path(markdown_file_path).parent / relative_file_path  # e.g., /path/to/markdown/media/image.png
        file_suffix = match.group(3)

        # print(f"Extracted and saved: {image_filepath}")
        if file_suffix == 'wmf': # in case of wmf, we need to convert it to svg with soffice
            svg_file_path = local_file_path.with_suffix('.svg') 
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(local_file_path),
                '-density', '300',
                '-trim', '-border', '5',
                str(svg_file_path),
            ], check=True)
            png_file_path = local_file_path.with_suffix('.png')
            relative_file_path = relative_file_path.with_suffix('.png')  # Update relative path to png
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(local_file_path),
                '-density', '300',
                '-trim', '-border', '5',
                str(png_file_path),
            ], check=True)
        elif file_suffix == 'emf': # in case of wmf, we need to convert it to svg with soffice
            svg_file_path = local_file_path.with_suffix('.svg') 
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(local_file_path),
                '-density', '300',
                '-trim', '-border', '5',
                str(svg_file_path),
            ], check=True)
            png_file_path = local_file_path.with_suffix('.png')
            relative_file_path = relative_file_path.with_suffix('.png')  # Update relative path to png
            subprocess.run([
                'magick', '-units', 'PixelsPerInch',
                str(local_file_path),
                '-density', '300',
                '-trim', '-border', '5',
                str(png_file_path),
            ], check=True)

        # Replace the original base64 string with the new link in the `new_content`
        # Only replace the first occurrence of this specific full_match_str in case of duplicates
        # (though each match from finditer is unique in its position)
        if png_file_path or svg_file_path:
            images_extracted_count += 1
        
        new_file_path = PurePosixPath(relative_file_path)
        
        new_content = new_content.replace(absolute_file_path, str(new_file_path), 1)  # replace wmf/emf absolute file path with the new png/svg relative file path if exists

    # if images_extracted_count == 0:
    #     return print("No wmf or emf image found in the Markdown file.")
    # output_markdown_file_path = markdown_file_path # Overwrite
    # output_markdown_file_path = os.path.splitext(markdown_file_path)[0] + "_modified.md" # New file

    with open(markdown_file_path, 'w', encoding='utf-8') as f:  # Overwrite the original file
        f.write(new_content)
    # print(f"Modified Markdown saved to: {output_markdown_file_path}, processed {images_extracted_count} image(s).")
    return images_extracted_count



# Optional: extract_md_html_images_win(Path("path/to/document.md"))

In [ ]:

# --- How to use it ---
if False:
    # Create a dummy Markdown file for testing
    dummy_md_content = """
# My Document

This is some text.

Here is an image: ![A red dot](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAUAAAAFCAYAAACNbyblAAAAHElEQVQI12P4//8/w38GIAXDIBKE0DHxgljNBAAO9TXL0Y4OHwAAAABJRU5ErkJggg==)

Some more text.

And another one: ![A blue square](data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAYABgAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjL/wAARCAAFAAUDASIAAhEBAxEB/8QAFQABAQAAAAAAAAAAAAAAAAAAAAb/xAAgEAACAQMEAwAAAAAAAAAAAAABAgADBBESBSFBUSKR/8QAFAEBAAAAAAAAAAAAAAAAAAAAAP/EABQRAQAAAAAAAAAAAAAAAAAAAAD/2gAMAwEAAhEDEQA/AIXVAvLBYy2PSkKOwz0A9YgA//Z)

This one is tricky with potential newlines in base64:
![With Newlines](data:image/gif;base64,R0lGODlhAQABAIAAAP///wAAACH5BAEAAAAALAAAAAABAAEAAAICRAEAOw==
)

End of document.
    """
    test_md_file = "test_document.md"
    with open(test_md_file, "w", encoding="utf-8") as f:
        f.write(dummy_md_content)
    print(f"Created dummy Markdown file: {test_md_file}")

    # Specify the path to your Markdown file
    markdown_file = test_md_file  # Or "your_actual_file.md"
    # Specify the folder (relative to the MD file) where images will be saved
    image_folder = "md_images"

    extract_base64_images(markdown_file, image_folder)

    # --- Optional: Clean up dummy files and folder after testing ---
    print("\nCleaning up dummy files...")
    if os.path.exists(os.path.join(os.path.dirname(test_md_file), image_folder)):
        for img_file in os.listdir(os.path.join(os.path.dirname(test_md_file), image_folder)):
            os.remove(os.path.join(os.path.dirname(test_md_file), image_folder, img_file))
        os.rmdir(os.path.join(os.path.dirname(test_md_file), image_folder))
    if os.path.exists(test_md_file):
        os.remove(test_md_file)
    print("Cleanup complete.")

In [ ]:
#| export
def convert_html_wmf_emf_image_from_md(root_folder):
    """
    Recursively convert all .md files under
    root_folder (and subfolders) and extract all base64 images in them into
    a separate folder and replace the base64 image references in the markdown
    files with the path to the extracted image.
    """


    root = Path(root_folder)
    for file in root.rglob('*'):
        if file.suffix.lower() == '.md':
            try:
                # Create a folder with the file name and move the md file into it
                # Use replace original md file with base64 extracted in separate image folder
                res = extract_md_html_images_win(file)
                if res == -1:
                    print(f"Not Converted: {file}")
                else:
                    print(f'Converted {res} images in {file}')
                # file.unlink()  # Remove the original md file after extraction
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {file}: {e}")
    

# Optional: convert_html_wmf_emf_image_from_md(PROCESS_REPORT["conversion"]["output_root"])

In [ ]:
#| export
def _extension_for_image_mime(mime: str) -> str:
    """Return a safe filename extension for an image MIME subtype."""
    normalized = mime.lower()
    if normalized in IMAGE_EXTENSION_BY_MIME:
        return IMAGE_EXTENSION_BY_MIME[normalized]
    fallback = re.sub(r"[^a-z0-9]+", "", normalized.split("+", 1)[0])[:16]
    return f".{fallback or 'bin'}"


def extract_md_base64_images_win(
    markdown_file_path: Path | str,
    image_output_folder: Path | str = "img",
) -> int:
    """
    Save embedded Markdown/HTML data-URI images and replace them with links.

    Image links always use forward slashes so the rewritten Markdown remains
    portable even though the notebook is intended to run on Windows.
    """
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        raise FileNotFoundError(f"Markdown file not found: {markdown_file}")

    relative_image_folder = Path(image_output_folder)
    if relative_image_folder.is_absolute() or ".." in relative_image_folder.parts:
        raise ValueError("image_output_folder must be relative to the Markdown file")

    image_directory = markdown_file.parent / relative_image_folder
    markdown_image_folder = PurePosixPath(relative_image_folder.as_posix())
    content = markdown_file.read_text(encoding="utf-8")
    images_extracted = 0

    def replace_data_image(match: re.Match) -> str:
        nonlocal images_extracted
        payload = "".join(match.group("data").split())
        payload += "=" * (-len(payload) % 4)
        try:
            image_data = base64.b64decode(payload, validate=True)
        except (binascii.Error, ValueError) as error:
            print(f"Invalid base64 image left unchanged in {markdown_file}: {error}")
            return match.group(0)

        images_extracted += 1
        extension = _extension_for_image_mime(match.group("mime"))
        image_filename = f"image-{images_extracted:04d}{extension}"
        image_directory.mkdir(parents=True, exist_ok=True)
        (image_directory / image_filename).write_bytes(image_data)

        markdown_image_path = (markdown_image_folder / image_filename).as_posix()
        suffix = match.group("suffix").lstrip()
        return f'{match.group("prefix")}{markdown_image_path}{suffix}'

    rewritten = MARKDOWN_DATA_IMAGE_RE.sub(replace_data_image, content)
    rewritten = HTML_DATA_IMAGE_RE.sub(replace_data_image, rewritten)
    if images_extracted:
        markdown_file.write_text(rewritten, encoding="utf-8")

    return images_extracted


def extract_base64_from_md(
    root_folder: Path | str,
    image_output_folder: Path | str = "img",
) -> dict[str, list | int]:
    """Recursively extract base64 images from every Markdown file below root."""
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Markdown root is not a directory: {root}")

    markdown_files = sorted(path for path in root.rglob("*.md") if path.is_file())
    report = {"processed": [], "failed": [], "images_extracted": 0}
    for markdown_file in markdown_files:
        try:
            count = extract_md_base64_images_win(markdown_file, image_output_folder)
        except (OSError, ValueError) as error:
            report["failed"].append((markdown_file, error))
            print(f"Failed to extract images from {markdown_file}: {error}")
            continue

        report["processed"].append(markdown_file)
        report["images_extracted"] += count
        print(f"Extracted {count} image(s): {markdown_file}")

    return report


def process_office_files(
    root_folder: Path | str | None = None,
    *,
    overwrite: bool = False,
    image_output_folder: Path | str = "img",
) -> dict[str, object]:
    """Run recursive Office conversion and base64 image extraction."""
    office_root = get_office_files_root() if root_folder is None else Path(root_folder)
    conversion = convert_office_to_md(office_root, overwrite=overwrite)
    extraction = extract_base64_from_md(
        conversion["output_root"],
        image_output_folder=image_output_folder,
    )
    return {"conversion": conversion, "extraction": extraction}

In [ ]:
# Run the complete workflow using OFFICE_FILES_ROOT from PROJ_ROOT/.env.
PROCESS_REPORT = process_office_files(overwrite=False, image_output_folder="img")
{
    "office_files_found": len(PROCESS_REPORT["conversion"]["discovered"]),
    "markdown_converted": len(PROCESS_REPORT["conversion"]["converted"]),
    "markdown_skipped": len(PROCESS_REPORT["conversion"]["skipped"]),
    "conversion_failures": len(PROCESS_REPORT["conversion"]["failed"]),
    "images_extracted": PROCESS_REPORT["extraction"]["images_extracted"],
    "extraction_failures": len(PROCESS_REPORT["extraction"]["failed"]),
    "output_root": str(PROCESS_REPORT["conversion"]["output_root"]),
}

In [ ]:
#| export
def copy_md_files(src_md_root: Path, dst_md_root: Path, bOverwrite: bool = True):
    """
    Recursively copy all .md files under
    src_md_folder (higher quality of original office converted md by MID) (and subfolders) 
    to dst_md_folder (low quality of original pdf converted md by gemini 2.5 pro exp).
    """

    # Create the destination folder if it does not exist
    if not dst_md_root.exists():
        dst_md_root.mkdir(parents=True, exist_ok=False)
    for file in src_md_root.rglob('*'):
        if file.suffix.lower() == '.md':
            try:
                # Create a folder with the file name and move the md file into it
                src_md_folder = file.parent
                dst_md_folder = dst_md_root / src_md_folder.relative_to(src_md_root) 
                if dst_md_folder.exists():
                    if bOverwrite:
                        shutil.rmtree(dst_md_folder)
                        print(f"Remove: {dst_md_folder}")
                    else:
                        print(f"Skipped: {dst_md_folder}")
                        continue
                shutil.copytree(src_md_folder, dst_md_folder)
                print(f"Copied: {src_md_folder} -> {dst_md_folder}")
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {file}: {e}")

In [ ]:
# Optional copy_md_files helper retained above; no hard-coded paths are run.

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

<div>
<link rel="stylesheet" href="https://gradio.s3-us-west-2.amazonaws.com/2.6.5/static/bundle.css">
<div id="target"></div>
<script src="https://gradio.s3-us-west-2.amazonaws.com/2.6.5/static/bundle.js"></script>
<script>
launchGradioFromSpaces("abidlabs/question-answering", "#target")
</script>
</div>